# (C) Telescope Resource Catalog — Independent Normalisation Verification

This notebook independently verifies the persisted Telescope Resource Catalog against its three construction inputs. It imports no producer code, writes no data product and reports explicit expected/observed evidence.

## Purpose and inputs

The verification reads only the frozen ICARE telescope and instrument tables, the curated external capability CSV and the persisted catalog. SHA-256 hashes are recorded before any measurement.

In [1]:
from pathlib import Path
import hashlib
import json
import math

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 120)

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/interim/telescopes").is_dir())
PATHS = {
    "telescopes": ROOT / "data/interim/telescopes/capture_20260808_071334/telescopes.parquet",
    "instruments": ROOT / "data/interim/telescopes/capture_20260808_071334/instruments.parquet",
    "external": ROOT / "data/raw/reference/telescope_external_capabilities.csv",
    "catalog": ROOT / "data/telescope_catalog/resource_catalog.parquet",
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

hashes_before = {name: sha256_file(path) for name, path in PATHS.items()}
telescopes = pd.read_parquet(PATHS["telescopes"])
instruments = pd.read_parquet(PATHS["instruments"])
external_raw = pd.read_csv(PATHS["external"], dtype=str, keep_default_na=False)
catalog = pd.read_parquet(PATHS["catalog"])

print("Files read:")
for name, path in PATHS.items():
    print(f"  {name:11s} {path.relative_to(ROOT)}  sha256={hashes_before[name]}")

Files read:
  telescopes  data/interim/telescopes/capture_20260808_071334/telescopes.parquet  sha256=40b1a37202c557b556309c1cf70771484de205e1f42d98e7e3d61bfee02b24d6
  instruments data/interim/telescopes/capture_20260808_071334/instruments.parquet  sha256=5680e911ae61a181c4b3001790e470e665e8152ebbaefce1600d9e87b90cae02
  external    data/raw/reference/telescope_external_capabilities.csv  sha256=6064fb0b70057ccdafb8ac669f67fed59f3d23a63b6d2704a93e87a2e7f7b63c
  catalog     data/telescope_catalog/resource_catalog.parquet  sha256=af2fc49fccb7346c77182c8a54ded954f56e4c27a65c1dfafa8582a60ebec164


## Catalog unit and schema

One catalog row must represent one ICARE instrument associated with one ICARE telescope. The frozen snapshot fixes the expected counts and the agreed design fixes the complete 20-column schema.

In [2]:
EXPECTED_COLUMNS = [
    "telescope_id", "telescope_name", "latitude", "longitude", "elevation",
    "telescope_diameter", "instrument_id", "instrument_name", "instrument_type",
    "instrument_band", "filters",
    "mlim_mag", "mlim_filter", "mlim_exposure", "mlim_status", "mlim_source",
    "usage_class", "followup_eligible", "restriction_note", "provenance_note",
]
missing_columns = [column for column in EXPECTED_COLUMNS if column not in catalog.columns]
unexpected_columns = [column for column in catalog.columns if column not in EXPECTED_COLUMNS]
schema_rows = [{"column": column, "dtype": str(catalog[column].dtype)} for column in catalog.columns]
print(pd.DataFrame(schema_rows).to_string(index=False))
print(f"expected counts: telescopes=89, instruments=95, external=89, catalog=95")
print(f"observed counts: telescopes={len(telescopes)}, instruments={len(instruments)}, external={len(external_raw)}, catalog={len(catalog)}")
print("missing expected columns:", missing_columns)
print("unexpected columns:", unexpected_columns)

            column   dtype
      telescope_id   int64
    telescope_name  object
          latitude float64
         longitude float64
         elevation float64
telescope_diameter float64
     instrument_id   int64
   instrument_name  object
   instrument_type  object
   instrument_band  object
           filters  object
          mlim_mag float64
       mlim_filter  object
     mlim_exposure  object
       mlim_status  object
       mlim_source  object
       usage_class  object
 followup_eligible boolean
  restriction_note  object
   provenance_note  object
expected counts: telescopes=89, instruments=95, external=89, catalog=95
observed counts: telescopes=89, instruments=95, external=89, catalog=95
missing expected columns: []
unexpected columns: []


## ICARE identity integrity

Identity is verified with native IDs and relationships, not names alone. Telescope attributes are taken from the telescope table and instrument attributes from the instrument table.

In [3]:
telescope_reference = telescopes[["id", "name", "lat", "lon", "elevation", "diameter"]].rename(columns={
    "id": "telescope_id", "name": "telescope_name", "lat": "latitude", "lon": "longitude",
    "diameter": "telescope_diameter"
})
instrument_reference = instruments[["id", "telescope_id", "name", "type", "band", "filters"]].rename(columns={
    "id": "instrument_id", "name": "instrument_name", "type": "instrument_type", "band": "instrument_band"
})
native_expected = instrument_reference.merge(telescope_reference, on="telescope_id", how="left", validate="many_to_one")
catalog_ids = set(catalog["instrument_id"])
instrument_ids = set(instruments["id"])
identity_join = catalog.merge(native_expected, on="instrument_id", how="outer", suffixes=("_catalog", "_icare"), indicator=True)
relationship_mismatches = identity_join["telescope_id_catalog"].ne(identity_join["telescope_id_icare"])

identity_evidence = pd.DataFrame([
    ["ICARE instruments", 95, len(instruments)],
    ["catalog rows", 95, len(catalog)],
    ["distinct catalog instrument IDs", 95, catalog["instrument_id"].nunique()],
    ["duplicate telescope/instrument pairs", 0, int(catalog.duplicated(["telescope_id", "instrument_id"]).sum())],
    ["catalog IDs outside ICARE", 0, len(catalog_ids - instrument_ids)],
    ["catalog telescope IDs outside ICARE", 0, len(set(catalog["telescope_id"]) - set(telescopes["id"]))],
    ["instrument-to-telescope relationship mismatches", 0, int(relationship_mismatches.sum())],
], columns=["measure", "expected", "observed"])
identity_evidence["status"] = identity_evidence["expected"].eq(identity_evidence["observed"]).map({True: "PASS", False: "FAIL"})
print(identity_evidence.to_string(index=False))
if relationship_mismatches.any():
    display(identity_join.loc[relationship_mismatches])

                                        measure  expected  observed status
                              ICARE instruments        95        95   PASS
                                   catalog rows        95        95   PASS
                distinct catalog instrument IDs        95        95   PASS
           duplicate telescope/instrument pairs         0         0   PASS
                      catalog IDs outside ICARE         0         0   PASS
            catalog telescope IDs outside ICARE         0         0   PASS
instrument-to-telescope relationship mismatches         0         0   PASS


## External capability matching

External rows are matched with stripped, case-folded telescope and instrument names only. Exact matches are measured separately, and the ICARE-only set is derived by a left anti-join.

In [4]:
def match_key(value):
    return str(value).strip().casefold()

external = external_raw.copy()
external["telescope_key"] = external["icare_telescope"].map(match_key)
external["instrument_key"] = external["icare_instrument"].map(match_key)
native_expected["telescope_key"] = native_expected["telescope_name"].map(match_key)
native_expected["instrument_key"] = native_expected["instrument_name"].map(match_key)
catalog_keyed = catalog.copy()
catalog_keyed["telescope_key"] = catalog_keyed["telescope_name"].map(match_key)
catalog_keyed["instrument_key"] = catalog_keyed["instrument_name"].map(match_key)
KEYS = ["telescope_key", "instrument_key"]
icare_key_set = set(map(tuple, native_expected[KEYS].to_numpy()))
external_key_set = set(map(tuple, external[KEYS].to_numpy()))
exact_icare_keys = set(zip(native_expected["telescope_name"], native_expected["instrument_name"]))
exact_match_count = sum(pair in exact_icare_keys for pair in zip(external["icare_telescope"], external["icare_instrument"]))
normalized_match_count = sum(pair in icare_key_set for pair in map(tuple, external[KEYS].to_numpy()))
unmatched_external = external[~external[KEYS].apply(tuple, axis=1).isin(icare_key_set)]
normalized_only = external[
    external[KEYS].apply(tuple, axis=1).isin(icare_key_set)
    & ~pd.Series(list(zip(external["icare_telescope"], external["icare_instrument"])), index=external.index).isin(exact_icare_keys)
]
icare_only = native_expected[~native_expected[KEYS].apply(tuple, axis=1).isin(external_key_set)].copy()
expected_icare_only_names = {
    ("ShAO-T2m", "Spectrograph UAGS"), ("GMG-2.4m", "YFOSC"),
    ("ShAO-T2m", "Spectrograph Canberra"), ("Xinglong-2.16m", "BFOSC"),
    ("EP-FXT", "EP-FXT"), ("SVOM", "MXT"),
}
derived_icare_only_names = set(zip(icare_only["telescope_name"], icare_only["instrument_name"]))
icare_only_catalog = catalog[catalog["instrument_id"].isin(icare_only["instrument_id"])][[
    "telescope_name", "instrument_name", "instrument_type", "mlim_status",
    "followup_eligible", "restriction_note"
]].sort_values(["telescope_name", "instrument_name"])
print(f"external rows: expected=89 observed={len(external)}")
print(f"unique normalized keys: expected=89 observed={external[KEYS].drop_duplicates().shape[0]}")
print(f"exact matches: expected=88 observed={exact_match_count}")
print(f"normalized matches: expected=89 observed={normalized_match_count}")
print(f"unmatched rows: expected=0 observed={len(unmatched_external)}")
print("Normalized-but-not-exact match:")
display(normalized_only[["icare_telescope", "icare_instrument", *KEYS]])
print("ICARE-only instruments derived by anti-join:")
display(icare_only_catalog)

external rows: expected=89 observed=89
unique normalized keys: expected=89 observed=89
exact matches: expected=88 observed=88
normalized matches: expected=89 observed=89
unmatched rows: expected=0 observed=0
Normalized-but-not-exact match:


,icare_telescope,icare_instrument,telescope_key,instrument_key
40,NUTTelA-TAO,NUTTelA-TAO,nuttela-tao,nuttela-tao


ICARE-only instruments derived by anti-join:


,telescope_name,instrument_name,instrument_type,mlim_status,followup_eligible,restriction_note
79,EP-FXT,EP-FXT,imager,UNKNOWN,False,Outside targeted optical photometric/imaging scope.
3,GMG-2.4m,YFOSC,spectrograph,UNKNOWN,False,Outside targeted optical photometric/imaging scope.
83,SVOM,MXT,imager,UNKNOWN,False,Outside targeted optical photometric/imaging scope.
5,ShAO-T2m,Spectrograph Canberra,spectrograph,UNKNOWN,False,Outside targeted optical photometric/imaging scope.
0,ShAO-T2m,Spectrograph UAGS,spectrograph,UNKNOWN,False,Outside targeted optical photometric/imaging scope.
16,Xinglong-2.16m,BFOSC,spectrograph,UNKNOWN,False,Outside targeted optical photometric/imaging scope.


## Native field preservation

Every native identity, type, band, filter, coordinate and diameter value is compared by instrument ID. Telescope and instrument names follow the documented outer-whitespace trim policy; band casing and all other native scientific values are preserved without semantic normalization.

In [5]:
def is_missing(value):
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False

def canonical_filter(value):
    if is_missing(value):
        return None
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
        except json.JSONDecodeError:
            return value
        return json.dumps(parsed, ensure_ascii=False, separators=(",", ":"))
    if isinstance(value, (list, tuple)):
        return json.dumps(list(value), ensure_ascii=False, separators=(",", ":"))
    return value

def cells_equal(left, right, column=None):
    if is_missing(left) and is_missing(right):
        return True
    if is_missing(left) or is_missing(right):
        return False
    if column == "filters":
        return left == right
    if column in {"telescope_name", "instrument_name", "instrument_type"}:
        return str(left).strip() == str(right).strip()
    if isinstance(left, (int, float)) and not isinstance(left, bool) and isinstance(right, (int, float)) and not isinstance(right, bool):
        return math.isclose(float(left), float(right), rel_tol=0.0, abs_tol=1e-12)
    return left == right

native_columns = ["telescope_id", "telescope_name", "instrument_name", "instrument_type", "instrument_band", "filters", "latitude", "longitude", "elevation", "telescope_diameter"]
native_join = catalog.merge(native_expected[["instrument_id", *native_columns]], on="instrument_id", validate="one_to_one", suffixes=("_catalog", "_icare"))
native_report_rows, native_mismatch_rows = [], []
for column in native_columns:
    equal = [cells_equal(a, b, column) for a, b in zip(native_join[f"{column}_catalog"], native_join[f"{column}_icare"])]
    unequal = len(equal) - sum(equal)
    native_report_rows.append({"column": column, "rows compared": len(equal), "equal": sum(equal), "unequal": unequal})
    for index in native_join.index[[not value for value in equal]]:
        native_mismatch_rows.append({"instrument_id": native_join.at[index, "instrument_id"], "column": column, "catalog": native_join.at[index, f"{column}_catalog"], "ICARE": native_join.at[index, f"{column}_icare"]})
native_report = pd.DataFrame(native_report_rows)
native_mismatches = pd.DataFrame(native_mismatch_rows, columns=["instrument_id", "column", "catalog", "ICARE"])
native_whitespace_differences = []
for column in ["telescope_name", "instrument_name", "instrument_type"]:
    raw_difference = native_join[f"{column}_catalog"].ne(native_join[f"{column}_icare"])
    for index in native_join.index[raw_difference]:
        native_whitespace_differences.append({"instrument_id": native_join.at[index, "instrument_id"], "column": column, "catalog": native_join.at[index, f"{column}_catalog"], "ICARE raw": native_join.at[index, f"{column}_icare"], "explanation": "surrounding whitespace stripped"})
print(native_report.to_string(index=False))
print(f"filters preserved: {native_report.set_index('column').at['filters', 'equal']} / 95")
print("Explained native text representation differences:")
display(pd.DataFrame(native_whitespace_differences))
if not native_mismatches.empty:
    display(native_mismatches)

            column  rows compared  equal  unequal
      telescope_id             95     95        0
    telescope_name             95     95        0
   instrument_name             95     95        0
   instrument_type             95     95        0
   instrument_band             95     95        0
           filters             95     95        0
          latitude             95     95        0
         longitude             95     95        0
         elevation             95     95        0
telescope_diameter             95     95        0
filters preserved: 95 / 95
Explained native text representation differences:


,instrument_id,column,catalog,ICARE raw,explanation
0,106,telescope_name,NUTTelA-TAO,NUTTelA-TAO,surrounding whitespace stripped
1,106,instrument_name,NUTTelA-TAO,NUTTelA-TAO,surrounding whitespace stripped


## Limiting-magnitude verification

The four external Mlim fields are compared at value level after only numeric parsing and ordinary blank-to-null conversion. All UNKNOWN rows are shown.

In [6]:
def optional_text(value):
    if is_missing(value):
        return None
    text = str(value).strip()
    return text if text else None

external_reference = external.copy()
external_reference["mlim_mag"] = pd.to_numeric(external_reference["mlim_mag"].str.strip().replace("", pd.NA), errors="coerce").astype("float64")
for column in ["mlim_filter", "mlim_exposure", "mlim_source", "usage_class", "restriction_note", "provenance_note"]:
    external_reference[column] = external_reference[column].map(optional_text)
external_reference["followup_eligible"] = external_reference["followup_eligible"].str.strip().str.casefold().map({"true": True, "false": False}).astype("boolean")
external_comparison = catalog_keyed.merge(external_reference, on=KEYS, how="inner", validate="one_to_one", suffixes=("_catalog", "_external"))

def value_report(frame, columns):
    report, mismatch_rows = [], []
    for column in columns:
        equal = [cells_equal(a, b, column) for a, b in zip(frame[f"{column}_catalog"], frame[f"{column}_external"])]
        report.append({"column": column, "rows compared": len(equal), "equal": sum(equal), "unequal": len(equal) - sum(equal)})
        for index in frame.index[[not value for value in equal]]:
            mismatch_rows.append({"instrument_id": frame.at[index, "instrument_id"], "column": column, "catalog": frame.at[index, f"{column}_catalog"], "external": frame.at[index, f"{column}_external"]})
    return pd.DataFrame(report), pd.DataFrame(mismatch_rows, columns=["instrument_id", "column", "catalog", "external"])

mlim_columns = ["mlim_mag", "mlim_filter", "mlim_exposure", "mlim_source"]
mlim_report, mlim_mismatches = value_report(external_comparison, mlim_columns)
known_mask = catalog["mlim_status"].eq("KNOWN")
unknown_mask = catalog["mlim_status"].eq("UNKNOWN")
def catalog_resource(telescope_name, instrument_name):
    rows = catalog[
        catalog["telescope_name"].map(match_key).eq(match_key(telescope_name))
        & catalog["instrument_name"].map(match_key).eq(match_key(instrument_name))
    ]
    if len(rows) != 1:
        raise RuntimeError(f"Expected one row for {telescope_name}/{instrument_name}; found {len(rows)}")
    return rows.iloc[0]

tarot_tre = catalog_resource("TAROT/TRE", "TAROT/TRE")
sedm = catalog_resource("Palomar 1.5m", "SEDM")
gmos = catalog_resource("Gemini North", "GMOS")
salt = catalog_resource("SALT", "SALT")
skynet = catalog_resource("SKYNET", "SKYNET/Prompt")
tarot_tre_ok = (
    tarot_tre["mlim_mag"] == 18.0
    and tarot_tre["mlim_filter"] == "ps1::open"
    and tarot_tre["mlim_exposure"] == "30 s"
    and "native icare sensitivity" in str(tarot_tre["provenance_note"]).casefold()
)
confirmed_unknown_ok = all(
    pd.isna(row["mlim_mag"]) and row["mlim_status"] == "UNKNOWN"
    for row in [sedm, gmos, salt]
)
skynet_ok = skynet["mlim_mag"] == 19.0 and "representative network/group" in str(skynet["provenance_note"]).casefold()
hdr_external = external_reference[external_reference["mlim_source"].eq("GRANDMA_HDR_2026")]
hdr_context_rows = hdr_external[hdr_external["mlim_filter"].notna()]
unknown_rows = catalog.loc[unknown_mask].sort_values(["telescope_name", "instrument_name"])
print(mlim_report.to_string(index=False))
print(f"KNOWN: expected=77 observed={int(known_mask.sum())}")
print(f"UNKNOWN: expected=18 observed={int(unknown_mask.sum())}")
print(f"TAROT/TRE native tuple verified: {tarot_tre_ok}")
print(f"SEDM/GMOS/SALT UNKNOWN verified: {confirmed_unknown_ok}")
print(f"SKYNET representative 19.0 verified: {skynet_ok}")
print(f"HDR rows with retained Mlim-filter context derived from CSV: {len(hdr_context_rows)}")
print("Complete UNKNOWN set:")
display(unknown_rows[["telescope_name", "instrument_name", "mlim_mag", "mlim_status", "followup_eligible", "restriction_note"]])
if not mlim_mismatches.empty:
    display(mlim_mismatches)

       column  rows compared  equal  unequal
     mlim_mag             89     89        0
  mlim_filter             89     89        0
mlim_exposure             89     89        0
  mlim_source             89     89        0
KNOWN: expected=77 observed=77
UNKNOWN: expected=18 observed=18
TAROT/TRE native tuple verified: True
SEDM/GMOS/SALT UNKNOWN verified: True
SKYNET representative 19.0 verified: True
HDR rows with retained Mlim-filter context derived from CSV: 9
Complete UNKNOWN set:


,telescope_name,instrument_name,mlim_mag,mlim_status,followup_eligible,restriction_note
2,Asteroid Terrestrial-impact Last Alert System,ATLAS,NaN,UNKNOWN,True,No accepted nominal limiting magnitude is retained.
23,CAHA-CAFOS,CAFOS,NaN,UNKNOWN,True,No photometric Mlim consolidated. The HDR value 20 is in the spectroscopy section and is intentionally not used as i...
79,EP-FXT,EP-FXT,NaN,UNKNOWN,False,Outside targeted optical photometric/imaging scope.
86,GCN,GCN,NaN,UNKNOWN,False,Generic/non-physical ICARE entry; not an observing resource.
21,GMG-2.4m,GMG-2.4,NaN,UNKNOWN,False,"Scope conflict: HDR lists GMG-2.4 under photometry, while ICARE types this instrument as spectrograph. Outside targe..."
3,GMG-2.4m,YFOSC,NaN,UNKNOWN,False,Outside targeted optical photometric/imaging scope.
73,Gemini North,GMOS,NaN,UNKNOWN,True,Imaging spectrograph; no spectroscopic threshold is stored as photometric Mlim.
66,Palomar 1.5m,SEDM,NaN,UNKNOWN,True,None
94,SALT,SALT,NaN,UNKNOWN,True,ICARE represents this resource as an optical imager; no spectroscopic threshold is stored as photometric Mlim.
83,SVOM,MXT,NaN,UNKNOWN,False,Outside targeted optical photometric/imaging scope.


## Static eligibility and restrictions

Curated static fields are compared directly. `followup_eligible` means static suitability for targeted optical photometric/imaging follow-up, not current availability. For the six ICARE-only instruments, the expected treatment is derived from native type/band evidence: clearly spectroscopy-only and X-ray/high-energy resources remain present but outside that scope.

In [7]:
static_columns = ["usage_class", "followup_eligible", "restriction_note", "provenance_note"]
static_report, static_mismatches = value_report(external_comparison, static_columns)
type_text = icare_only["instrument_type"].fillna("").astype(str).str.strip().str.casefold()
band_text = icare_only["instrument_band"].fillna("").astype(str).str.strip().str.casefold()
spectroscopic_only = type_text.str.contains("spectro") & ~type_text.str.contains("imaging")
high_energy = band_text.str.replace("-", "", regex=False).str.replace(" ", "", regex=False).str.contains("xray") | band_text.str.contains("gamma")
outside_scope_ids = set(icare_only.loc[spectroscopic_only | high_energy, "instrument_id"])
icare_only_persisted = catalog[catalog["instrument_id"].isin(icare_only["instrument_id"])].copy()
icare_only_policy_ok = (
    outside_scope_ids == set(icare_only["instrument_id"])
    and icare_only_persisted["followup_eligible"].eq(False).all()
    and icare_only_persisted["restriction_note"].eq("Outside targeted optical photometric/imaging scope.").all()
)
eligible_count = int(catalog["followup_eligible"].eq(True).sum())
ineligible_count = int(catalog["followup_eligible"].eq(False).sum())
corrected_static_pairs = [
    ("Gemini North", "GMOS"), ("SALT", "SALT"), ("SVOM", "VT"),
    ("Swift", "UVOTXRT"), ("VIRT", "VIRT"), ("Zadko", "Zadko"),
]
corrected_static_rows = [catalog_resource(telescope_name, instrument_name) for telescope_name, instrument_name in corrected_static_pairs]
corrected_static_resources_ok = all(row["followup_eligible"] == True for row in corrected_static_rows)
virt_row = catalog_resource("VIRT", "VIRT")
zadko_row = catalog_resource("Zadko", "Zadko")
temporary_status_not_boolean_rule = (
    virt_row["followup_eligible"] == True
    and zadko_row["followup_eligible"] == True
    and "historical availability" in str(virt_row["restriction_note"]).casefold()
    and "historical availability" in str(zadko_row["restriction_note"]).casefold()
)
ineligible_rows = catalog[catalog["followup_eligible"].eq(False)].sort_values(["telescope_name", "instrument_name"])
print(static_report.to_string(index=False))
print(f"eligible: expected=83 observed={eligible_count}")
print(f"ineligible: expected=12 observed={ineligible_count}")
print(f"six corrected instrument-level eligibility values true: {corrected_static_resources_ok}")
print(f"time-specific availability excluded from the boolean rule: {temporary_status_not_boolean_rule}")
print("Complete ineligible set:")
display(ineligible_rows[["telescope_name", "instrument_name", "instrument_type", "mlim_status", "restriction_note"]])
if not static_mismatches.empty:
    display(static_mismatches)

           column  rows compared  equal  unequal
      usage_class             89     89        0
followup_eligible             89     89        0
 restriction_note             89     89        0
  provenance_note             89     89        0
eligible: expected=83 observed=83
ineligible: expected=12 observed=12
six corrected instrument-level eligibility values true: True
time-specific availability excluded from the boolean rule: True
Complete ineligible set:


,telescope_name,instrument_name,instrument_type,mlim_status,restriction_note
79,EP-FXT,EP-FXT,imager,UNKNOWN,Outside targeted optical photometric/imaging scope.
86,GCN,GCN,imager,UNKNOWN,Generic/non-physical ICARE entry; not an observing resource.
21,GMG-2.4m,GMG-2.4,spectrograph,UNKNOWN,"Scope conflict: HDR lists GMG-2.4 under photometry, while ICARE types this instrument as spectrograph. Outside targe..."
3,GMG-2.4m,YFOSC,spectrograph,UNKNOWN,Outside targeted optical photometric/imaging scope.
52,PAN-STARRS 1.8m,PS1,imager,KNOWN,Non-repointable survey resource.
82,Rubin Observatory Simonyi Survey Telescope,LSST,imager,KNOWN,Non-repointable survey resource.
83,SVOM,MXT,imager,UNKNOWN,Outside targeted optical photometric/imaging scope.
5,ShAO-T2m,Spectrograph Canberra,spectrograph,UNKNOWN,Outside targeted optical photometric/imaging scope.
0,ShAO-T2m,Spectrograph UAGS,spectrograph,UNKNOWN,Outside targeted optical photometric/imaging scope.
53,TAROT,TAROT,imager,UNKNOWN,"Generic TAROT network entry; use TAROT/TCA, TAROT/TCH and TAROT/TRE instead."


## Missing-value semantics

Missing Mlim must be a numeric null with status UNKNOWN. It must not become a zero, string sentinel or automatic reason for ineligibility.

In [8]:
missing_mlim = catalog["mlim_mag"].isna()
known_semantics_ok = catalog.loc[known_mask, "mlim_mag"].notna().all()
unknown_semantics_ok = (missing_mlim.equals(unknown_mask) and catalog.loc[unknown_mask, "mlim_mag"].isna().all())
no_zero_sentinel = not catalog.loc[unknown_mask, "mlim_mag"].eq(0).any()
numeric_mlim_dtype = pd.api.types.is_numeric_dtype(catalog["mlim_mag"])
unknown_but_eligible = int((unknown_mask & catalog["followup_eligible"].eq(True)).sum())
missing_external_mlim = external_reference["mlim_mag"].isna()
external_missing_preserved = external_comparison.loc[external_comparison["mlim_mag_external"].isna(), "mlim_mag_catalog"].isna().all()
null_evidence = pd.DataFrame([
    ["KNOWN has numeric Mlim", True, bool(known_semantics_ok)],
    ["UNKNOWN equals missing Mlim set", True, bool(unknown_semantics_ok)],
    ["no zero sentinel among UNKNOWN", True, bool(no_zero_sentinel)],
    ["mlim_mag has numeric dtype", True, bool(numeric_mlim_dtype)],
    ["external missing Mlim remains null", True, bool(external_missing_preserved)],
    ["UNKNOWN rows still eligible", "> 0", unknown_but_eligible],
], columns=["measure", "expected", "observed"])
print(null_evidence.to_string(index=False))

                           measure expected observed
            KNOWN has numeric Mlim     True     True
   UNKNOWN equals missing Mlim set     True     True
    no zero sentinel among UNKNOWN     True     True
        mlim_mag has numeric dtype     True     True
external missing Mlim remains null     True     True
       UNKNOWN rows still eligible      > 0        8


## Scope exclusions

The schema is inspected for event-dependent, FoV, allocation and historical-observation fields. The catalog describes resources, not events or live operations.

In [9]:
column_keys = {column: column.casefold() for column in catalog.columns}
dynamic_tokens = ["observable", "observability", "detectable", "detectability", "source_magnitude", "event_magnitude", "airmass", "weather", "current_availability", "ranking", "recommendation", "llm_output"]
fov_tokens = ["fov", "field_of_view", "region"]
allocation_tokens = ["allocation", "proposal_id", "group_id", "hours_allocated", "validity_range"]
historical_tokens = ["observation_id", "obstime", "seeing", "target_name", "processed_fraction", "historical_limmag"]
def fields_containing(tokens):
    return [column for column, key in column_keys.items() if any(token in key for token in tokens)]

dynamic_fields = fields_containing(dynamic_tokens)
fov_fields = fields_containing(fov_tokens)
allocation_fields = fields_containing(allocation_tokens)
historical_fields = fields_containing(historical_tokens)
scope_evidence = pd.DataFrame([
    ["dynamic/event-specific fields", [], dynamic_fields],
    ["FoV fields", [], fov_fields],
    ["allocation fields", [], allocation_fields],
    ["historical observation fields", [], historical_fields],
], columns=["category", "expected", "observed"])
scope_evidence["status"] = scope_evidence["observed"].map(lambda values: "PASS" if not values else "FAIL")
print(scope_evidence.to_string(index=False))

                     category expected observed status
dynamic/event-specific fields       []       []   PASS
                   FoV fields       []       []   PASS
            allocation fields       []       []   PASS
historical observation fields       []       []   PASS


## Independent value-level reconstruction

An expected table is reconstructed from the three inputs using the documented decisions only: ICARE defines the 95-row universe and native fields, external data are left-joined by minimal normalized names, missing external rows retain null capability metadata, and clearly outside-scope native types receive the static restriction. All 20 persisted columns are then compared independently by instrument ID.

In [10]:
external_payload = external_reference[[*KEYS, "mlim_mag", "mlim_filter", "mlim_exposure", "mlim_source", "usage_class", "followup_eligible", "restriction_note", "provenance_note"]]
expected = native_expected.merge(external_payload, on=KEYS, how="left", validate="one_to_one", indicator="external_match")
has_external = expected["external_match"].eq("both")
type_key = expected["instrument_type"].fillna("").astype(str).str.strip().str.casefold()
band_key = expected["instrument_band"].fillna("").astype(str).str.strip().str.casefold()
clear_spectrograph = type_key.str.contains("spectro") & ~type_key.str.contains("imaging")
compact_band = band_key.str.replace("-", "", regex=False).str.replace(" ", "", regex=False)
clear_high_energy = compact_band.str.contains("xray") | band_key.str.contains("gamma")
native_outside_scope = ~has_external & (clear_spectrograph | clear_high_energy)
expected["followup_eligible"] = expected["followup_eligible"].astype("boolean")
expected.loc[native_outside_scope, "followup_eligible"] = False
expected.loc[native_outside_scope, "restriction_note"] = "Outside targeted optical photometric/imaging scope."
expected["mlim_status"] = expected["mlim_mag"].notna().map({True: "KNOWN", False: "UNKNOWN"})
for column in ["telescope_name", "instrument_name", "instrument_type", "instrument_band", "mlim_filter", "mlim_exposure", "mlim_status", "mlim_source", "usage_class", "restriction_note", "provenance_note"]:
    expected[column] = expected[column].map(optional_text)
expected = expected[EXPECTED_COLUMNS].copy()
expected["telescope_id"] = expected["telescope_id"].astype("int64")
expected["instrument_id"] = expected["instrument_id"].astype("int64")
expected["telescope_diameter"] = expected["telescope_diameter"].astype("float64")
expected["mlim_mag"] = expected["mlim_mag"].astype("float64")
expected["followup_eligible"] = expected["followup_eligible"].astype("boolean")

full_join = catalog.merge(expected, on="instrument_id", how="outer", validate="one_to_one", suffixes=("_catalog", "_expected"), indicator=True)
full_report_rows, full_mismatch_rows = [], []
for column in EXPECTED_COLUMNS:
    if column == "instrument_id":
        equal = full_join["_merge"].eq("both").tolist()
    else:
        equal = [cells_equal(a, b, column) for a, b in zip(full_join[f"{column}_catalog"], full_join[f"{column}_expected"])]
    full_report_rows.append({"column": column, "rows compared": len(equal), "equal": sum(equal), "unequal": len(equal) - sum(equal)})
    for index in full_join.index[[not value for value in equal]]:
        full_mismatch_rows.append({"instrument_id": full_join.at[index, "instrument_id"], "column": column, "catalog": full_join.at[index, f"{column}_catalog"] if column != "instrument_id" else full_join.at[index, "_merge"], "expected": full_join.at[index, f"{column}_expected"] if column != "instrument_id" else "both"})
full_report = pd.DataFrame(full_report_rows)
full_mismatches = pd.DataFrame(full_mismatch_rows, columns=["instrument_id", "column", "catalog", "expected"])
print(full_report.to_string(index=False))
print(f"columns compared: {len(full_report)}")
print(f"total mismatches: {int(full_report['unequal'].sum())}")
if not full_mismatches.empty:
    display(full_mismatches)

            column  rows compared  equal  unequal
      telescope_id             95     95        0
    telescope_name             95     95        0
          latitude             95     95        0
         longitude             95     95        0
         elevation             95     95        0
telescope_diameter             95     95        0
     instrument_id             95     95        0
   instrument_name             95     95        0
   instrument_type             95     95        0
   instrument_band             95     95        0
           filters             95     95        0
          mlim_mag             95     95        0
       mlim_filter             95     95        0
     mlim_exposure             95     95        0
       mlim_status             95     95        0
       mlim_source             95     95        0
       usage_class             95     95        0
 followup_eligible             95     95        0
  restriction_note             95     95        0


## Provenance

Native ICARE identity is checked separately from external fields. The two concise external provenance fields are verified directly, and ICARE-only rows must not receive fabricated external provenance.

In [11]:
provenance_report = static_report[static_report["column"].eq("provenance_note")].copy()
mlim_source_report = mlim_report[mlim_report["column"].eq("mlim_source")].copy()
icare_only_no_fabricated_provenance = icare_only_persisted[["mlim_source", "provenance_note"]].isna().all().all()
provenance_ok = (
    "mlim_source" in catalog.columns
    and "provenance_note" in catalog.columns
    and int(provenance_report["unequal"].sum()) == 0
    and int(mlim_source_report["unequal"].sum()) == 0
    and bool(icare_only_no_fabricated_provenance)
    and int(native_report["unequal"].sum()) == 0
)
provenance_evidence = pd.DataFrame([
    ["native ICARE values preserved", 0, int(native_report["unequal"].sum())],
    ["external mlim_source mismatches", 0, int(mlim_source_report["unequal"].sum())],
    ["external provenance_note mismatches", 0, int(provenance_report["unequal"].sum())],
    ["ICARE-only rows with fabricated provenance", 0, int((~icare_only_persisted[["mlim_source", "provenance_note"]].isna()).any(axis=1).sum())],
], columns=["measure", "expected", "observed"])
provenance_evidence["status"] = provenance_evidence["expected"].eq(provenance_evidence["observed"]).map({True: "PASS", False: "FAIL"})
print(provenance_evidence.to_string(index=False))

                                   measure  expected  observed status
             native ICARE values preserved         0         0   PASS
           external mlim_source mismatches         0         0   PASS
       external provenance_note mismatches         0         0   PASS
ICARE-only rows with fabricated provenance         0         0   PASS


## Final verification summary

The final matrix consolidates the independent measurements. Source hashes are recomputed last, after all notebook work. Every status is explicitly PASS or FAIL.

In [12]:
hashes_after = {name: sha256_file(path) for name, path in PATHS.items()}
hashes_unchanged = hashes_before == hashes_after
name_mismatches = int(native_report.loc[native_report["column"].isin(["telescope_name", "instrument_name"]), "unequal"].sum())
type_mismatches = int(native_report.loc[native_report["column"].eq("instrument_type"), "unequal"].sum())
filter_mismatches = int(native_report.loc[native_report["column"].eq("filters"), "unequal"].sum())
coordinate_mismatches = int(native_report.loc[native_report["column"].isin(["latitude", "longitude", "elevation"]), "unequal"].sum())
diameter_mismatches = int(native_report.loc[native_report["column"].eq("telescope_diameter"), "unequal"].sum())
band_mismatches = int(native_report.loc[native_report["column"].eq("instrument_band"), "unequal"].sum())
mlim_value_mismatches = int(mlim_report["unequal"].sum())
eligibility_value_mismatches = int(static_report["unequal"].sum())
followup_external_mismatches = int(static_report.loc[static_report["column"].eq("followup_eligible"), "unequal"].sum())
context_mismatches = int(mlim_report["unequal"].sum() + static_report["unequal"].sum())
full_mismatch_count = int(full_report["unequal"].sum())

matrix_rows = []
def record(number, check, expected_value, observed_value, passed):
    matrix_rows.append({"number": f"{number:02d}", "check": check, "expected": str(expected_value), "observed": str(observed_value), "status": "PASS" if bool(passed) else "FAIL"})

record(1, "ICARE instrument count", 95, len(instruments), len(instruments) == 95)
record(2, "catalog row count", 95, len(catalog), len(catalog) == 95)
record(3, "unique instrument IDs", "95 unique, each once", f"{catalog['instrument_id'].nunique()} unique; duplicates={int(catalog['instrument_id'].duplicated().sum())}", catalog["instrument_id"].nunique() == 95 and not catalog["instrument_id"].duplicated().any())
record(4, "unique telescope/instrument pairs", 95, catalog[["telescope_id", "instrument_id"]].drop_duplicates().shape[0], not catalog.duplicated(["telescope_id", "instrument_id"]).any())
record(5, "all catalog IDs originate from ICARE", "0 outside", len(catalog_ids - instrument_ids), not (catalog_ids - instrument_ids))
record(6, "telescope relationships preserved", "0 mismatches", int(relationship_mismatches.sum()), not relationship_mismatches.any())
record(7, "canonical names preserved under trim policy", "0 unexplained mismatches after outer-whitespace trimming", name_mismatches, name_mismatches == 0)
record(8, "instrument types preserved", "0 mismatches", type_mismatches, type_mismatches == 0)
record(9, "filters preserved", "95/95", f"{95-filter_mismatches}/95", filter_mismatches == 0)
record(10, "telescope coordinates preserved", "0 mismatches across latitude/longitude/elevation", coordinate_mismatches, coordinate_mismatches == 0)
record(11, "telescope_diameter preserved", "95/95 native values", f"{95-diameter_mismatches}/95", diameter_mismatches == 0)
record(12, "telescope_diameter completeness", "95/95 final rows", f"{catalog['telescope_diameter'].notna().sum()}/95", catalog["telescope_diameter"].notna().sum() == 95)
record(13, "instrument_band preserved", "95/95 native values with casing unchanged", f"{95-band_mismatches}/95", band_mismatches == 0)
record(14, "instrument_band completeness", "95/95 final rows", f"{catalog['instrument_band'].notna().sum()}/95", catalog["instrument_band"].notna().sum() == 95)
record(15, "external rows unique", "89 rows; 89 keys", f"{len(external)} rows; {external[KEYS].drop_duplicates().shape[0]} keys", len(external) == 89 and external[KEYS].drop_duplicates().shape[0] == 89)
record(16, "all external rows matched", "89 matched; 0 unmatched", f"{normalized_match_count} matched; {len(unmatched_external)} unmatched", normalized_match_count == 89 and unmatched_external.empty)
record(17, "six ICARE-only instruments retained", "derived set=6 and all retained", f"derived={len(icare_only)}; retained={len(icare_only_persisted)}", len(icare_only) == 6 and derived_icare_only_names == expected_icare_only_names and len(icare_only_persisted) == 6)
record(18, "corrected TAROT/TRE tuple and SKYNET representative value", "TAROT/TRE=18,ps1::open,30 s,native provenance; SKYNET=19.0 representative", f"TAROT/TRE={tarot_tre_ok}; SKYNET={skynet_ok}", tarot_tre_ok and skynet_ok)
record(19, "SEDM Mlim UNKNOWN", "null/UNKNOWN", f"mlim={sedm['mlim_mag']}; status={sedm['mlim_status']}", pd.isna(sedm["mlim_mag"]) and sedm["mlim_status"] == "UNKNOWN")
record(20, "GMOS Mlim UNKNOWN", "null/UNKNOWN", f"mlim={gmos['mlim_mag']}; status={gmos['mlim_status']}", pd.isna(gmos["mlim_mag"]) and gmos["mlim_status"] == "UNKNOWN")
record(21, "SALT Mlim UNKNOWN", "null/UNKNOWN", f"mlim={salt['mlim_mag']}; status={salt['mlim_status']}", pd.isna(salt["mlim_mag"]) and salt["mlim_status"] == "UNKNOWN")
record(22, "Mlim KNOWN semantics", "all KNOWN numeric", bool(known_semantics_ok), bool(known_semantics_ok))
record(23, "Mlim UNKNOWN semantics", "all and only null; no sentinel", f"null-set-equal={unknown_semantics_ok}; no-zero={no_zero_sentinel}; numeric-dtype={numeric_mlim_dtype}", unknown_semantics_ok and no_zero_sentinel and numeric_mlim_dtype)
record(24, "Mlim counts", "KNOWN=77; UNKNOWN=18", f"KNOWN={int(known_mask.sum())}; UNKNOWN={int(unknown_mask.sum())}", int(known_mask.sum()) == 77 and int(unknown_mask.sum()) == 18)
record(25, "static eligibility values preserved", "0 external mismatches; six corrected resources true; ICARE-only policy consistent", f"external={followup_external_mismatches}; corrected={corrected_static_resources_ok}; ICARE-only={icare_only_policy_ok}", followup_external_mismatches == 0 and corrected_static_resources_ok and icare_only_policy_ok)
record(26, "eligibility counts", "eligible=83; ineligible=12", f"eligible={eligible_count}; ineligible={ineligible_count}", eligible_count == 83 and ineligible_count == 12)
record(27, "temporary operational state not encoded as eligibility rule", "VIRT/Zadko true; historical notes separate", temporary_status_not_boolean_rule, temporary_status_not_boolean_rule)
record(28, "provenance and context fields preserved", "0 external context mismatches; HDR context derived from CSV; no fabrication", f"mismatches={context_mismatches}; HDR context rows={len(hdr_context_rows)}; provenance={provenance_ok}", context_mismatches == 0 and len(hdr_context_rows) == 9 and provenance_ok and confirmed_unknown_ok)
record(29, "no FoV/event/dynamic fields", "none", f"FoV={fov_fields}; dynamic={dynamic_fields}; allocations={allocation_fields}; observations={historical_fields}", not fov_fields and not dynamic_fields and not allocation_fields and not historical_fields)
record(30, "exact expected 20-column schema", "20 columns; missing=[]; unexpected=[]", f"{len(catalog.columns)} columns; missing={missing_columns}; unexpected={unexpected_columns}", list(catalog.columns) == EXPECTED_COLUMNS and len(catalog.columns) == 20 and not missing_columns and not unexpected_columns)
record(31, "independent full value comparison", "20 columns; 0 mismatches", f"{len(full_report)} columns; {full_mismatch_count} mismatches", len(full_report) == 20 and full_mismatch_count == 0)
record(32, "source and final hashes unchanged", "all four unchanged", f"{sum(hashes_before[name] == hashes_after[name] for name in PATHS)}/4 unchanged", hashes_unchanged)
verification = pd.DataFrame(matrix_rows)[["number", "check", "expected", "observed", "status"]]
print(verification.to_string(index=False))
pass_count = int(verification["status"].eq("PASS").sum())
fail_count = int(verification["status"].eq("FAIL").sum())

print("\n" + "=" * 60)
print("TELESCOPE RESOURCE CATALOG — INDEPENDENT VERIFICATION")
print("=" * 60)
print(f"\nInputs:\n    ICARE telescopes: {len(telescopes)}\n    ICARE instruments: {len(instruments)}\n    external rows: {len(external)}\n    catalog rows: {len(catalog)}")
print(f"\nIdentity:\n    unique instrument IDs: {catalog['instrument_id'].nunique()}\n    relationship mismatches: {int(relationship_mismatches.sum())}")
print(f"\nExternal matching:\n    exact: {exact_match_count}\n    normalized: {normalized_match_count}\n    unmatched: {len(unmatched_external)}\n    ICARE-only rows: {len(icare_only)}")
print(f"\nMlim:\n    KNOWN: {int(known_mask.sum())}\n    UNKNOWN: {int(unknown_mask.sum())}\n    value mismatches: {mlim_value_mismatches}")
print(f"\nFollow-up eligibility:\n    eligible: {eligible_count}\n    ineligible: {ineligible_count}\n    value mismatches: {eligibility_value_mismatches}")
print(f"\nFull value comparison:\n    columns compared: {len(full_report)}\n    total mismatches: {full_mismatch_count}")
print(f"\nSchema:\n    columns: {len(catalog.columns)}\n    missing: {missing_columns}\n    unexpected: {unexpected_columns}")
print(f"\nSource integrity:\n    unchanged inputs/output: {sum(hashes_before[name] == hashes_after[name] for name in PATHS)}/4")
print(f"\nVerification:\n    {pass_count} PASS / {fail_count} FAIL")
print(f"\nNotebook:\n    cells: 25\n    execution errors: 0")
print(f"\nOVERALL:\n    {'PASS' if fail_count == 0 else 'FAIL'}")
if fail_count:
    raise RuntimeError("Independent verification failed:\n" + verification.loc[verification["status"].eq("FAIL")].to_string(index=False))

number                                                       check                                                                          expected                                              observed status
    01                                      ICARE instrument count                                                                                95                                                    95   PASS
    02                                           catalog row count                                                                                95                                                    95   PASS
    03                                       unique instrument IDs                                                              95 unique, each once                               95 unique; duplicates=0   PASS
    04                           unique telescope/instrument pairs                                                                                95            